# Prepare CSV for Parashar

See [`DailyNotes.md`](../../DailyNotes.md#L766)

Requested fields are stored in different CSV outputs across [`output/`](../../output/)

Accession is considered scBaseCount ID

Field | CSV
--- | ---
CyteType Run ID | [`job_details.csv`](../../output/cytetype_pipeline/20260522_175813/job_details.csv)
Accession | All
Number of clusters | [`run.csv`](../../output/clustering_pipeline/20260519_160627/run.csv) (clustering pipeline run summary)
Disease classification | [`datasets_subset_qc.csv`](../../output/metadata/datasets_subset_qc.csv)
Name of `obs` entry in `h5ad` containing `group_key` | `leiden_merged` (same for all h5ad files)

## Build CSV

### Load CSVs before joining

In [ ]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("../..").resolve()

CYTETYPE_JOB_DETAILS = REPO_ROOT / "output/cytetype_pipeline/20260522_175813/job_details.csv"
CLUSTERING_RUN = REPO_ROOT / "output/clustering_pipeline/20260519_160627/run.csv"
DATASETS_SUBSET_QC = REPO_ROOT / "output/metadata/datasets_subset_qc.csv"

job_details = pd.read_csv(CYTETYPE_JOB_DETAILS)
clustering_run = pd.read_csv(CLUSTERING_RUN)
datasets_subset_qc = pd.read_csv(DATASETS_SUBSET_QC)

job_details.head()

### Join CSVs by accession (`srx` or `srx_accession`)

In [ ]:
df = (
    job_details.merge(clustering_run, on="srx", how="inner", suffixes=("_cytetype", "_clustering"))
    .merge(
        datasets_subset_qc,
        left_on="srx",
        right_on="srx_accession",
        how="inner",
    )
)

len(df), len(job_details)

### Drop unwanted columns, add `group_key` column

In [ ]:
GROUP_KEY = "leiden_merged"

df = df.assign(group_key=GROUP_KEY)[  # pyright: ignore[reportCallIssue]
    [
        "job_id",
        "srx",
        "nClustersPostMerge",
        "diseaseLabel",
        "group_key",
    ]
].rename(
    columns={
        "job_id": "cytetype_run_id",
        "srx": "accession",
        "nClustersPostMerge": "n_clusters",
        "diseaseLabel": "disease_classification",
    }
)

df.head()

### Save CSV

In [ ]:
OUTPUT_PATH = REPO_ROOT / "output/extra/cytetype_runs_for_parashar.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH